### Student C: Deep Learning (MLP) + Comparison
### Project 16: Credit Card Fraud Detection

#### Imports 

In [23]:
import sys
!"{sys.executable}" -m pip install numpy pandas matplotlib torch scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report, RocCurveDisplay, PrecisionRecallDisplay)
import os, time

#### Configration

In [25]:
RESULTS_DIR = "D:\\Queens\\DL-CISC 867-Deep Learning\\DL Project\\credit-card-fraud-detection\\results"
MODELS_DIR  = "D:\\Queens\\DL-CISC 867-Deep Learning\\DL Project\\credit-card-fraud-detection\\models"
SEED        = 42
EPOCHS      = 50
BATCH_SIZE  = 1024
LR          = 1e-3
os.makedirs(MODELS_DIR, exist_ok=True)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")

---
#### Load Data

In [26]:
X_train = np.load(f"{RESULTS_DIR}/X_train_sm.npy").astype(np.float32)
y_train = np.load(f"{RESULTS_DIR}/y_train_sm.npy").astype(np.float32)
X_val   = np.load(f"{RESULTS_DIR}/X_val.npy").astype(np.float32)
y_val   = np.load(f"{RESULTS_DIR}/y_val.npy").astype(np.float32)
X_test  = np.load(f"{RESULTS_DIR}/X_test.npy").astype(np.float32)
y_test  = np.load(f"{RESULTS_DIR}/y_test.npy").astype(np.float32)

# PyTorch DataLoaders
def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(X_train, y_train)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)

---
#### MLP Architecture

In [27]:
#### MLP Architecture
class FraudMLP(nn.Module):
    def __init__(self, input_dim=30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1),          nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

model     = FraudMLP(input_dim=X_train.shape[1]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# Class-weighted loss
fraud_weight = (y_train == 0).sum() / (y_train == 1).sum()
criterion = nn.BCELoss(reduction='none')

def weighted_bce(pred, target, pos_weight=fraud_weight):
    loss = criterion(pred, target)
    weights = torch.where(target == 1,
        torch.tensor(pos_weight, dtype=torch.float32),
        torch.tensor(1.0, dtype=torch.float32))
    return (loss * weights).mean()

def focal_loss(pred, target, gamma=2.0, alpha=0.25):
    bce = nn.functional.binary_cross_entropy(pred, target, reduction='none')
    pt = torch.where(target == 1, pred, 1 - pred)
    focal_weight = alpha * (1 - pt) ** gamma
    return (focal_weight * bce).mean()

#### Deeper MLP

In [28]:
class FraudMLP5Layer(nn.Module):
    def __init__(self, input_dim=30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1),          nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

---
#### Training Loop

In [29]:
train_losses, val_losses, val_pr_aucs = [], [], []
best_pr_auc, best_epoch = 0, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_losses = []
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(Xb)
        loss = weighted_bce(pred, yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    # Validation
    model.eval()
    val_probs, val_true = [], []
    with torch.no_grad():
        vl = 0
        for Xb, yb in val_loader:
            pred = model(Xb.to(DEVICE))
            vl += weighted_bce(pred, yb.to(DEVICE)).item()
            val_probs.extend(pred.cpu().numpy())
            val_true.extend(yb.numpy())

    epoch_val_loss = vl / len(val_loader)
    pr_auc = average_precision_score(val_true, val_probs)
    scheduler.step(epoch_val_loss)

    train_losses.append(np.mean(batch_losses))
    val_losses.append(epoch_val_loss)
    val_pr_aucs.append(pr_auc)

    if pr_auc > best_pr_auc:
        best_pr_auc, best_epoch = pr_auc, epoch
        torch.save(model.state_dict(), f"{MODELS_DIR}/mlp_best.pt")

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_losses[-1]:.4f} "
              f"| Val Loss: {epoch_val_loss:.4f} | Val PR-AUC: {pr_auc:.4f}")   
print(f"\nBest Val PR-AUC: {best_pr_auc:.4f} at epoch {best_epoch}")

Epoch  10/50 | Train Loss: 0.0074 | Val Loss: 0.0080 | Val PR-AUC: 0.7826
Epoch  20/50 | Train Loss: 0.0035 | Val Loss: 0.0077 | Val PR-AUC: 0.7646
Epoch  30/50 | Train Loss: 0.0020 | Val Loss: 0.0080 | Val PR-AUC: 0.7676
Epoch  40/50 | Train Loss: 0.0019 | Val Loss: 0.0082 | Val PR-AUC: 0.7707
Epoch  50/50 | Train Loss: 0.0018 | Val Loss: 0.0083 | Val PR-AUC: 0.7705

Best Val PR-AUC: 0.7826 at epoch 10


---
#### Learning Curves

In [30]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses,   label='Val Loss')
ax1.set_title("MLP Training & Validation Loss")
ax1.set_xlabel("Epoch"); ax1.legend()
ax2.plot(val_pr_aucs, color='green')
ax2.axvline(best_epoch - 1, linestyle='--', color='red', label=f'Best (E{best_epoch})')
ax2.set_title("Validation PR-AUC"); ax2.set_xlabel("Epoch"); ax2.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/mlp_learning_curves.png", dpi=150)
plt.close()

---
#### Test Evaluation

In [31]:
model.load_state_dict(torch.load(f"{MODELS_DIR}/mlp_best.pt"))
model.eval()
with torch.no_grad():
    test_probs = model(torch.FloatTensor(X_test).to(DEVICE)).cpu().numpy()
test_preds = (test_probs >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, test_probs)
pr_auc  = average_precision_score(y_test, test_probs)
print(f"\nTest ROC-AUC: {roc_auc:.4f} | Test PR-AUC: {pr_auc:.4f}")
print(classification_report(y_test, test_preds, target_names=['Legit', 'Fraud']))


Test ROC-AUC: 0.9767 | Test PR-AUC: 0.8319
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.46      0.89      0.60        98

    accuracy                           1.00     56962
   macro avg       0.73      0.94      0.80     56962
weighted avg       1.00      1.00      1.00     56962



---
#### Ablation Study 

In [34]:
ablation_results = []

for config_name, ModelClass, loss_fn in configs:
    print(f"\nTraining: {config_name}")
    
    # Fresh model + optimizer for each config
    abl_model     = ModelClass(input_dim=X_train.shape[1]).to(DEVICE)
    abl_optimizer = torch.optim.Adam(abl_model.parameters(), lr=LR, weight_decay=1e-5)
    abl_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(abl_optimizer, patience=5, factor=0.5)
    
    best_abl_pr, best_abl_epoch = 0, 0

    for epoch in range(1, EPOCHS + 1):
        # Train
        abl_model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            abl_optimizer.zero_grad()
            loss = loss_fn(abl_model(Xb), yb)
            loss.backward()
            abl_optimizer.step()

        # Validate
        abl_model.eval()
        val_probs, val_true, vl = [], [], 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                pred = abl_model(Xb.to(DEVICE))
                vl  += loss_fn(pred, yb.to(DEVICE)).item()
                val_probs.extend(pred.cpu().numpy())
                val_true.extend(yb.numpy())

        pr_auc = average_precision_score(val_true, val_probs)
        abl_scheduler.step(vl / len(val_loader))

        if pr_auc > best_abl_pr:
            best_abl_pr, best_abl_epoch = pr_auc, epoch
            torch.save(abl_model.state_dict(), f"{MODELS_DIR}/abl_{config_name.replace(' ', '_')}.pt")

    # Test evaluation with best checkpoint
    abl_model.load_state_dict(torch.load(f"{MODELS_DIR}/abl_{config_name.replace(' ', '_')}.pt"))
    abl_model.eval()
    with torch.no_grad():
        test_probs = abl_model(torch.FloatTensor(X_test).to(DEVICE)).cpu().numpy()

    test_roc = roc_auc_score(y_test, test_probs)
    test_pr  = average_precision_score(y_test, test_probs)

    print(f"  Best Val PR-AUC: {best_abl_pr:.4f} (epoch {best_abl_epoch}) | Test ROC-AUC: {test_roc:.4f} | Test PR-AUC: {test_pr:.4f}")
    ablation_results.append({
        "config":        config_name,
        "best_val_pr":   round(best_abl_pr,  4),
        "best_epoch":    best_abl_epoch,
        "test_roc_auc":  round(test_roc, 4),
        "test_pr_auc":   round(test_pr,  4)
    })

# Save to CSV
ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(f"{RESULTS_DIR}/ablation_summary.csv", index=False)
print(f"\nAblation Study Results:\n{ablation_df.to_string(index=False)}")


Training: 4-Layer + Weighted BCE
  Best Val PR-AUC: 0.7953 (epoch 30) | Test ROC-AUC: 0.9697 | Test PR-AUC: 0.8491

Training: 4-Layer + Focal Loss
  Best Val PR-AUC: 0.7841 (epoch 6) | Test ROC-AUC: 0.9781 | Test PR-AUC: 0.8104

Training: 5-Layer + Weighted BCE
  Best Val PR-AUC: 0.7766 (epoch 12) | Test ROC-AUC: 0.9758 | Test PR-AUC: 0.8593

Ablation Study Results:
                config  best_val_pr  best_epoch  test_roc_auc  test_pr_auc
4-Layer + Weighted BCE       0.7953          30        0.9697       0.8491
  4-Layer + Focal Loss       0.7841           6        0.9781       0.8104
5-Layer + Weighted BCE       0.7766          12        0.9758       0.8593


---
#### Final Comparison Across All Models

In [36]:
import joblib
classical    = pd.read_csv(f"{RESULTS_DIR}/classical_model_summary.csv")
ablation_df  = pd.read_csv(f"{RESULTS_DIR}/ablation_summary.csv")

# Main MLP result
mlp_row = pd.DataFrame([{"name": "MLP (4-Layer + Weighted BCE)", "roc_auc": roc_auc, "pr_auc": pr_auc}])

# Ablation rows — rename columns to match
abl_rows = ablation_df[["config", "test_roc_auc", "test_pr_auc"]].rename(columns={
    "config":       "name",
    "test_roc_auc": "roc_auc",
    "test_pr_auc":  "pr_auc"
})

all_models = pd.concat([classical[["name","roc_auc","pr_auc"]], mlp_row, abl_rows], ignore_index=True)
all_models.to_csv(f"{RESULTS_DIR}/full_model_comparison.csv", index=False)
print(f"\nFinal Model Comparison:\n{all_models.to_string(index=False)}")


Final Model Comparison:
                        name  roc_auc   pr_auc
         Logistic Regression 0.973080 0.725819
               Random Forest 0.986043 0.816518
                     XGBoost 0.978725 0.869984
MLP (4-Layer + Weighted BCE) 0.976654 0.752340
      4-Layer + Weighted BCE 0.969700 0.849100
        4-Layer + Focal Loss 0.978100 0.810400
      5-Layer + Weighted BCE 0.975800 0.859300


#### ───────The End───────
